# Test multilingual training pipeline
Notebook này kiểm tra từng phần mà không chạy toàn bộ training. Sau mỗi bước process data, một mẫu đã chuẩn hóa sẽ được in ra để kiểm tra.

In [5]:
from pathlib import Path
import json
from itertools import islice

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'src').exists():
    REPO_ROOT = REPO_ROOT.parent

MODEL_NAME = 'Qwen/Qwen2.5-0.5B'
LANGUAGE_PAIR = 'vi-en'
DIRECTION = 'both'
INSTRUCTION_LANGUAGE = 'vi'
MAX_LENGTH = 256
RUN_FULL_LENGTH_SCAN = True  # đổi thành True để quét toàn bộ split
LENGTH_SCAN_SPLITS = ('train',)  # có thể dùng ('train', 'valid', 'test')
print('Repository:', REPO_ROOT)

Repository: d:\NLP\Multilingual


## 1. Process Stage 1 data và xem mẫu đã xử lý

In [2]:
%cd ..

d:\NLP\Multilingual


In [3]:
from datasets import Dataset
from src.prepare_data import _discover_mt, _parallel_rows

train_paths = _discover_mt(REPO_ROOT / 'data' / 'MT', 'train', LANGUAGE_PAIR)
processed_rows = list(islice(_parallel_rows(train_paths, DIRECTION), 4))
stage1_sample_dataset = Dataset.from_list(processed_rows)

print('Files:', [str(path) for path in train_paths])
print('Number of sampled processed rows:', len(stage1_sample_dataset))
print('\nMẪU SAU KHI PROCESS_DATA:')
print(json.dumps(stage1_sample_dataset[0], ensure_ascii=False, indent=2))

Files: ['d:\\NLP\\Multilingual\\data\\MT\\vi-en\\train.vi-en.general_trans.json']
Number of sampled processed rows: 4

MẪU SAU KHI PROCESS_DATA:
{
  "source": "Năm 2015, ước tính có khoảng 5% người từ 15 đến 65 tuổi đã sử dụng thuốc bất hợp pháp ít nhất một lần (158 triệu đến 351 triệu người).",
  "target": "In 2015, it was estimated that about 5% of people aged 15 to 65 had used illegal drugs at least once (158 million to 351 million).",
  "source_lang": "vi",
  "target_lang": "en"
}


## 2. Kiểm tra plain-text translation prompt

In [9]:
from src.prompts import TRANSLATION_TARGET_MARKER, translation_instruction

sample = stage1_sample_dataset[0]
translation_prompt = (
    translation_instruction(sample['source_lang'], sample['target_lang'])
    + sample['source']
    + TRANSLATION_TARGET_MARKER
    + sample['target']
)
print(translation_prompt)

Translate from vi to en.
Source: Năm 2015, ước tính có khoảng 5% người từ 15 đến 65 tuổi đã sử dụng thuốc bất hợp pháp ít nhất một lần (158 triệu đến 351 triệu người).
Translation: In 2015, it was estimated that about 5% of people aged 15 to 65 had used illegal drugs at least once (158 million to 351 million).


## 3. Tokenize, collate và kiểm tra lại source/target spans

In [7]:
from transformers import AutoTokenizer
from src.collator import MultilingualDataCollator

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

alignment_collator = MultilingualDataCollator(
    tokenizer=tokenizer,
)
batch = alignment_collator(processed_rows[:2])
print({key: tuple(value.shape) for key, value in batch.items()})

{'input_ids': (2, 108), 'attention_mask': (2, 108), 'labels': (2, 108), 'source_start_positions': (2,), 'source_end_positions': (2,), 'target_start_positions': (2,), 'target_end_positions': (2,)}


### Quét độ dài lớn nhất của MT dataset (tùy chọn)
Đặt `RUN_FULL_LENGTH_SCAN=True` ở cell cấu hình. Dữ liệu được đọc streaming và tokenizer chạy theo batch; không giữ toàn bộ dataset trong RAM.

In [10]:
from tqdm.auto import tqdm

def chunked_rows(iterator, batch_size=256):
    chunk = []
    for row in iterator:
        chunk.append(row)
        if len(chunk) == batch_size:
            yield chunk
            chunk = []
    if chunk:
        yield chunk

if not RUN_FULL_LENGTH_SCAN:
    print('Bỏ qua full scan. Đặt RUN_FULL_LENGTH_SCAN=True để chạy.')
else:
    scan_paths = []
    for split_name in LENGTH_SCAN_SPLITS:
        scan_paths.extend(_discover_mt(
            REPO_ROOT / 'data' / 'MT', split_name, LANGUAGE_PAIR
        ))

    marker_length = len(tokenizer(
        TRANSLATION_TARGET_MARKER, add_special_tokens=False
    )['input_ids'])
    eos_length = int(tokenizer.eos_token_id is not None)
    instruction_length_cache = {}
    longest = None
    sample_count = 0

    row_iterator = _parallel_rows(scan_paths, DIRECTION)
    for row_batch in tqdm(chunked_rows(row_iterator), desc='Scanning token lengths'):
        source_tokens = tokenizer(
            [row['source'] for row in row_batch], add_special_tokens=False
        )['input_ids']
        target_tokens = tokenizer(
            [row['target'] for row in row_batch], add_special_tokens=False
        )['input_ids']

        for row, src_ids, tgt_ids in zip(row_batch, source_tokens, target_tokens):
            direction_key = (row['source_lang'], row['target_lang'])
            if direction_key not in instruction_length_cache:
                instruction_length_cache[direction_key] = len(tokenizer(
                    translation_instruction(*direction_key),
                    add_special_tokens=False,
                )['input_ids'])
            instruction_length = instruction_length_cache[direction_key]
            total_length = (
                instruction_length + len(src_ids) + marker_length
                + len(tgt_ids) + eos_length
            )
            sample_count += 1
            if longest is None or total_length > longest['total_tokens']:
                longest = {
                    'source_lang': row['source_lang'],
                    'target_lang': row['target_lang'],
                    'instruction_tokens': instruction_length,
                    'source_tokens': len(src_ids),
                    'marker_tokens': marker_length,
                    'target_tokens': len(tgt_ids),
                    'eos_tokens': eos_length,
                    'total_tokens': total_length,
                    'source_preview': row['source'][:300],
                    'target_preview': row['target'][:300],
                }

    print('Scanned samples:', sample_count)
    print('Tokenizer model_max_length:', tokenizer.model_max_length)
    print('Mẫu dài nhất:')
    print(json.dumps(longest, ensure_ascii=False, indent=2))
    if tokenizer.model_max_length < 10**9:
        print('Vượt context window:', longest['total_tokens'] > tokenizer.model_max_length)

Scanning token lengths: 0it [00:00, ?it/s]

Scanned samples: 200000
Tokenizer model_max_length: 131072
Mẫu dài nhất:
{
  "source_lang": "vi",
  "target_lang": "en",
  "instruction_tokens": 9,
  "source_tokens": 225,
  "marker_tokens": 4,
  "target_tokens": 220,
  "eos_tokens": 1,
  "total_tokens": 459,
  "source_preview": "Tỉ lệ lạm phát hàng năm,% 1994 1160.262 1995 60.388 1996 28.763 1997 11.321 1998 1.880 1999 18.095 2000 10.001 2001 6.582 2002 6.686 2003 7.001 2004 7.011 2005 7.868 2006 8.400 2007 18.772 2008 9.484 2009 6.377 2010 7.969 2011 7.429 2012 6.0",
  "target_preview": "Annual inflation rate,% 1994 1160.262 1995 60.388 1996 28.763 1997 11.321 1998 1.880 1999 18.095 2000 10.001 2001 6.582 2002 6.686 2003 7.001 2004 7.011 2005 7.868 2006 8.400 2007 18.772 2008 9.484 2009 6.377 2010 7.969 2011 7.429 2012 6.0"
}
Vượt context window: False


In [6]:
row_index = 0
src_start = batch['source_start_positions'][row_index].item()
src_end = batch['source_end_positions'][row_index].item()
tgt_start = batch['target_start_positions'][row_index].item()
tgt_end = batch['target_end_positions'][row_index].item()
ids = batch['input_ids'][row_index]

print('Full sequence:')
print(tokenizer.decode(ids, skip_special_tokens=False))
print('\nSource span:', (src_start, src_end))
print(tokenizer.decode(ids[src_start:src_end], skip_special_tokens=False))
print('\nTarget span:', (tgt_start, tgt_end))
print(tokenizer.decode(ids[tgt_start:tgt_end], skip_special_tokens=False))
supervised_ids = batch['labels'][row_index][batch['labels'][row_index] != -100]
print('\nTokens được tính NTP loss:')
print(tokenizer.decode(supervised_ids, skip_special_tokens=False))

Full sequence:
Translate from vi to en.
Source: Năm 2015, ước tính có khoảng 5% người từ 15 đến 65 tuổi đã sử dụng thuốc bất hợp pháp ít nhất một lần (158 triệu đến 351 triệu người).
Translation: In 2015, it was estimated that about 5% of people aged 15 to 65 had used illegal drugs at least once (158 million to 351 million).<|endoftext|>

Source span: (9, 59)
Năm 2015, ước tính có khoảng 5% người từ 15 đến 65 tuổi đã sử dụng thuốc bất hợp pháp ít nhất một lần (158 triệu đến 351 triệu người).

Target span: (63, 107)
In 2015, it was estimated that about 5% of people aged 15 to 65 had used illegal drugs at least once (158 million to 351 million).

Tokens được tính NTP loss:
In 2015, it was estimated that about 5% of people aged 15 to 65 had used illegal drugs at least once (158 million to 351 million).<|endoftext|>


## 4. Chạy thử model Stage 1
Cell này tải model và chạy một forward pass. Có thể bỏ qua nếu chỉ muốn kiểm tra data.

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model
from src.model import MultilingualAlignmentModel

model = MultilingualAlignmentModel(
    MODEL_NAME,
    contrastive_weight=0.1,
    ot_weight=0.05,
    align_layer=-1,
    attention_mass_weight=0.5,
    attn_implementation='eager',
)
model.lm = get_peft_model(model.lm, LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
))
model.lm.print_trainable_parameters()
output = model(**batch)
print('Total Stage 1 loss:', float(output.loss.detach()))
print('Logits shape:', tuple(output.logits.shape))

## 5. Mini train thử LoRA
Chạy ba optimizer steps trên sample nhỏ để kiểm tra backward, loss components và logging.

In [8]:
import importlib
from transformers import TrainingArguments
import src.train as train_module
importlib.reload(train_module)
ComponentLoggingTrainer = train_module.ComponentLoggingTrainer

from peft import LoraConfig, TaskType, get_peft_model
from src.model import MultilingualAlignmentModel

model = MultilingualAlignmentModel(
    MODEL_NAME,
    contrastive_weight=0.1,
    ot_weight=0.05,
    align_layer=-1,
    attention_mass_weight=0.5,
    attn_implementation='eager',
)
model.lm = get_peft_model(model.lm, LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
))
model.lm.print_trainable_parameters()
output = model(**batch)
print('Total Stage 1 loss:', float(output.loss.detach()))
print('Logits shape:', tuple(output.logits.shape))

del output  # release the previous forward graph before training
mini_training_args = TrainingArguments(
    output_dir=str(REPO_ROOT / 'outputs' / 'notebook-smoke-test'),
    max_steps=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=1,
    learning_rate=2e-4,
    logging_strategy='steps',
    logging_steps=1,
    logging_first_step=True,
    save_strategy='no',
    eval_strategy='steps',
    eval_steps=1,
    report_to='none',
    remove_unused_columns=False,
)
mini_trainer = ComponentLoggingTrainer(
    model=model,
    args=mini_training_args,
    stage='alignment',
    train_dataset=stage1_sample_dataset,
    eval_dataset=stage1_sample_dataset,  # smoke test: reuse the tiny sample
    data_collator=alignment_collator,
)
mini_result = mini_trainer.train()
print('Mini-train metrics:', mini_result.metrics)

import pandas as pd
component_history = [
    row for row in mini_trainer.state.log_history
    if any(key.startswith(('train/', 'eval/')) for key in row) or 'eval_loss' in row
]
display(pd.DataFrame(component_history))

trainable params: 2,162,688 || all params: 496,195,456 || trainable%: 0.4359
Total Stage 1 loss: 1.4398548603057861
Logits shape: (2, 108, 151936)


Step,Training Loss
1,1.867900
2,1.393300
3,1.309000


Loss components | train/trainer_loss=1.867854 | train/model_total_loss=1.867854 | train/ntp_loss=1.163638 | train/contrastive_loss=6.565183 | train/ot_loss=0.953942 | train/weighted_contrastive_loss=0.656518 | train/weighted_ot_loss=0.047697
Loss components | train/trainer_loss=1.393286 | train/model_total_loss=1.393286 | train/ntp_loss=0.688767 | train/contrastive_loss=6.598370 | train/ot_loss=0.893647 | train/weighted_contrastive_loss=0.659837 | train/weighted_ot_loss=0.044682
Loss components | train/trainer_loss=1.308985 | train/model_total_loss=1.308985 | train/ntp_loss=0.614033 | train/contrastive_loss=6.507461 | train/ot_loss=0.884124 | train/weighted_contrastive_loss=0.650746 | train/weighted_ot_loss=0.044206
Mini-train metrics: {'train_runtime': 3.2963, 'train_samples_per_second': 1.82, 'train_steps_per_second': 0.91, 'total_flos': 0.0, 'train_loss': 1.5233750343322754, 'epoch': 1.5}


,loss,grad_norm,learning_rate,train/trainer_loss,train/model_total_loss,train/ntp_loss,train/contrastive_loss,train/ot_loss,train/weighted_contrastive_loss,train/weighted_ot_loss,epoch,step
0,1.8679,4.969274,0.000200,1.867854,1.867854,1.163638,6.565183,0.953942,0.656518,0.047697,0.5,1
1,1.3933,2.851751,0.000133,1.393286,1.393286,0.688767,6.598370,0.893647,0.659837,0.044682,1.0,2
2,1.3090,2.561648,0.000067,1.308985,1.308985,0.614033,6.507461,0.884124,0.650746,0.044206,1.5,3


## 6. Process Stage 2 data và xem mẫu XLSum/Bactrian

In [ ]:
import ijson
from src.prompts import summarization_instruction

xlsum_path = REPO_ROOT / 'data' / 'XLSum' / 'XLSum' / INSTRUCTION_LANGUAGE / f'validation.{INSTRUCTION_LANGUAGE}.json'
with xlsum_path.open('r', encoding='utf-8') as handle:
    raw_xlsum = json.loads(next(line for line in handle if line.strip()))
processed_xlsum = {
    'instruction': summarization_instruction(INSTRUCTION_LANGUAGE),
    'input': raw_xlsum['text'].strip(),
    'output': raw_xlsum['summary'].strip(),
    'language': INSTRUCTION_LANGUAGE,
    'dataset_name': 'xlsum',
}
print('MẪU XLSUM SAU KHI PROCESS_DATA:')
print(json.dumps(processed_xlsum, ensure_ascii=False, indent=2))

In [ ]:
bactrian_path = REPO_ROOT / 'data' / 'Bactrian-Multilingual_Instruction' / f'{INSTRUCTION_LANGUAGE}.json'
with bactrian_path.open('rb') as handle:
    raw_bactrian = next(ijson.items(handle, 'item'))
processed_bactrian = {
    'instruction': str(raw_bactrian['instruction']).strip(),
    'input': str(raw_bactrian.get('input') or '').strip(),
    'output': str(raw_bactrian['output']).strip(),
    'language': INSTRUCTION_LANGUAGE,
    'dataset_name': 'bactrian',
}
print('MẪU BACTRIAN SAU KHI PROCESS_DATA:')
print(json.dumps(processed_bactrian, ensure_ascii=False, indent=2))

## 7. Kiểm tra Stage 2 prompt và response-only labels

In [ ]:
from src.collator import InstructionDataCollator

instruction_collator = InstructionDataCollator(tokenizer, max_length=MAX_LENGTH)
instruction_batch = instruction_collator([processed_xlsum, processed_bactrian])
row_index = 0
ids = instruction_batch['input_ids'][row_index]
labels = instruction_batch['labels'][row_index]
print('Full Stage 2 sequence:')
print(tokenizer.decode(ids, skip_special_tokens=False))
print('\nChỉ phần sau được tính NTP loss:')
print(tokenizer.decode(labels[labels != -100], skip_special_tokens=False))

## 8. Load full datasets (tùy chọn)
Chỉ chạy cell dưới khi muốn tạo toàn bộ Hugging Face Dataset cache.

In [ ]:
# from src.prepare_data import load_parallel_dataset, load_instruction_dataset
# full_stage1 = load_parallel_dataset(
#     data_dir=str(REPO_ROOT / 'data' / 'MT'),
#     language_pairs=LANGUAGE_PAIR,
#     direction=DIRECTION,
# )
# print(full_stage1)
# print(full_stage1['train'][0])
# full_stage2 = load_instruction_dataset(languages=INSTRUCTION_LANGUAGE)
# print(full_stage2)
# print(full_stage2['train'][0])

In [5]:
import json

vi_en_path = "data/MT/vi-en/train.vi-en.general_trans.json"
km_en_path = "data/MT/km-en/train.km-en.general_trans.json"
output_path = "data/MT/vi-en/train.1.vi-en.general_trans.json"

# 1. Lấy toàn bộ câu English từ km-en
km_en_sentences = set()

with open(km_en_path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue

        item = json.loads(line)
        en = item["translation"]["en"].strip()
        km_en_sentences.add(en)

# 2. Lọc vi-en
total = 0
removed = 0
kept = 0

with open(vi_en_path, "r", encoding="utf-8") as fin, \
     open(output_path, "w", encoding="utf-8") as fout:

    for line in fin:
        line = line.strip()
        if not line:
            continue

        item = json.loads(line)
        total += 1

        en = item["translation"]["en"].strip()

        if en in km_en_sentences:
            removed += 1
            continue

        fout.write(json.dumps(item, ensure_ascii=False) + "\n")
        kept += 1

print(f"Total vi-en : {total}")
print(f"Removed     : {removed}")
print(f"Kept        : {kept}")
print(f"Saved to    : {output_path}")

Total vi-en : 100106
Removed     : 50186
Kept        : 49920
Saved to    : data/MT/vi-en/train.1.vi-en.general_trans.json


In [6]:
import json
import re
import unicodedata


def clean_thai_text(text: str) -> str:
    # Unicode normalization
    text = unicodedata.normalize("NFC", text)

    # Xóa các ký tự vô hình / bidi không cần thiết
    text = re.sub(
        r"[\u200b\u200c\u200d\u200e\u200f"
        r"\u202a-\u202e\u2066-\u2069\ufeff]",
        "",
        text
    )

    # Normalize whitespace
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\s*\n\s*", " ", text)

    return text.strip()


input_path = "data/MT/th-en/train.th-en.general_trans.json"
output_path = "data/MT/th-en/train.1.th-en.general_trans.json"

total = 0
changed = 0

with open(input_path, "r", encoding="utf-8") as fin, \
     open(output_path, "w", encoding="utf-8") as fout:

    for line in fin:
        line = line.strip()
        if not line:
            continue

        item = json.loads(line)

        thai = item["translation"]["th"]
        cleaned = clean_thai_text(thai)

        if thai != cleaned:
            changed += 1

        item["translation"]["th"] = cleaned

        fout.write(
            json.dumps(item, ensure_ascii=False) + "\n"
        )

        total += 1

print(f"Total   : {total}")
print(f"Changed : {changed}")
print(f"Saved   : {output_path}")

Total   : 52309
Changed : 6646
Saved   : data/MT/th-en/train.1.th-en.general_trans.json
